# 双均线量化投资策略实验
## 策略原理
使用短期均线（5 日）和长期均线（20 日）的交叉信号：
- **金叉**：短均线上穿长均线 → 买入信号
- **死叉**：短均线下穿长均线 → 卖出信号

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 获取数据（以沪深 300ETF 为例）
ticker = '510300.SS'  # 沪深 300ETF
data = yf.download(ticker, start='2023-01-01', end='2024-12-01', progress=False)
print(f'数据获取成功：{len(data)} 个交易日')
print(data.head())

In [ ]:
# 计算均线
data['MA5'] = data['Close'].rolling(window=5).mean()
data['MA20'] = data['Close'].rolling(window=20).mean()

# 生成交易信号
data['Signal'] = 0
data.loc[data['MA5'] > data['MA20'], 'Signal'] = 1  # 持仓
data['Position'] = data['Signal'].diff()  # 交易点

print('均线计算完成')
print(data[['Close', 'MA5', 'MA20', 'Signal']].tail(10))

In [ ]:
# 可视化
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(data.index, data['Close'], label='收盘价', alpha=0.7)
ax.plot(data.index, data['MA5'], label='MA5', linewidth=1)
ax.plot(data.index, data['MA20'], label='MA20', linewidth=1)

# 标记买卖点
buy_signals = data[data['Position'] == 1]
sell_signals = data[data['Position'] == -1]
ax.scatter(buy_signals.index, buy_signals['MA5'], marker='^', color='r', s=100, label='买入', zorder=5)
ax.scatter(sell_signals.index, sell_signals['MA5'], marker='v', color='g', s=100, label='卖出', zorder=5)

ax.set_title('双均线策略 - 沪深 300ETF')
ax.set_xlabel('日期')
ax.set_ylabel('价格')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('strategy_chart.png', dpi=150)
plt.show()

In [ ]:
# 策略回测
initial_capital = 1000000  # 初始资金 100 万
data['Returns'] = data['Close'].pct_change()
data['Strategy_Returns'] = data['Signal'].shift(1) * data['Returns']

# 累计收益
data['Cumulative_Market'] = (1 + data['Returns']).cumprod()
data['Cumulative_Strategy'] = (1 + data['Strategy_Returns']).cumprod()

# 绩效指标
total_return = data['Cumulative_Strategy'].iloc[-1] - 1
market_return = data['Cumulative_Market'].iloc[-1] - 1
sharpe = np.sqrt(252) * data['Strategy_Returns'].mean() / data['Strategy_Returns'].std()
max_drawdown = (data['Cumulative_Strategy'] / data['Cumulative_Strategy'].cummax() - 1).min()

print('=' * 50)
print('策略回测结果')
print('=' * 50)
print(f'初始资金：¥{initial_capital:,.0f}')
print(f'策略累计收益：{total_return*100:.2f}%')
print(f'市场累计收益：{market_return*100:.2f}%')
print(f'夏普比率：{sharpe:.3f}')
print(f'最大回撤：{max_drawdown*100:.2f}%')
print('=' * 50)

In [ ]:
# 收益曲线对比
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(data.index, data['Cumulative_Market'], label='买入持有', alpha=0.7)
ax.plot(data.index, data['Cumulative_Strategy'], label='双均线策略', linewidth=2)
ax.set_title('策略收益 vs 市场收益')
ax.set_xlabel('日期')
ax.set_ylabel('累计收益倍数')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('performance_chart.png', dpi=150)
plt.show()